# iSCORS-Net — Fast Runner

**Workflow:**
1. Run **Setup** — clones the repo and installs deps.
2. Run **Train** — internal learning on the test video.
3. Run **Results** — inline visualisation.
4. Run **Download** — saves `results_<VERSION>.zip` to your machine.

---

In [ ]:
VERSION = 'v3.0'
print(f'iSCORS-Net {VERSION}')

In [ ]:
import os

REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'

if not os.path.isdir('iscors-net'):
    !git clone --depth 1 -b {BRANCH} {REPO}
else:
    !git -C iscors-net pull

os.chdir('iscors-net')
print('Working dir:', os.getcwd())

!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
import os

# Always regenerate — ensures new background design (near-static) is used
video_path = './data/test_synthetic_cell.tif'
if os.path.exists(video_path):
    os.remove(video_path)
    print('Removed old video.')

os.makedirs('./data', exist_ok=True)
!python utils/generate_test_video.py

In [ ]:
!python train_phys_recon.py

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)

---

## Real Cell Analysis

**Workflow:**
1. **real-setup** — Mount Google Drive, unzip `large_file.zip`, locate files.
2. **real-mat** — Decode `Output_iSCORS_map.mat` → TIF reference maps (not fed to model).
3. **real-preprocess** — Normalise video: ÷ temporal median → ÷ per-frame Gaussian (σ=4).
4. **real-inference** — Run trained model on preprocessed video.
5. **real-compare** — Side-by-side: model predictions vs traditional iSCORS reference.


In [ ]:
# ── Real Cell Analysis: Mount Drive & Extract ─────────────────────────────
from google.colab import drive
import zipfile, os

drive.mount('/content/drive', force_remount=False)

ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'
EXTRACT_DIR = '/content/real_data'
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)

def find_file(root, name):
    for dirpath, _, files in os.walk(root):
        if name in files:
            return os.path.join(dirpath, name)
    return None

# Skip extraction if key files are already present (avoids re-extracting every run)
VIDEO_PATH = find_file(EXTRACT_DIR, 'COBRI_rarw_video.tif')
MAT_PATH   = find_file(EXTRACT_DIR, 'Output_iSCORS_map.mat')

if VIDEO_PATH and MAT_PATH:
    print(f'Files already extracted — skipping zip extraction.')
else:
    print(f'Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print('Extraction done.')
    VIDEO_PATH = find_file(EXTRACT_DIR, 'COBRI_rarw_video.tif')
    MAT_PATH   = find_file(EXTRACT_DIR, 'Output_iSCORS_map.mat')

print(f'Video : {VIDEO_PATH}')
print(f'MAT   : {MAT_PATH}')
assert VIDEO_PATH, 'COBRI_rarw_video.tif not found in zip!'
assert MAT_PATH,   'Output_iSCORS_map.mat not found in zip!'


In [ ]:
# ── Decode Output_iSCORS_map.mat → TIF (reference only, not fed to model) ──
import numpy as np
import tifffile
import matplotlib.pyplot as plt

# Try scipy.io (MATLAB v5/v7); fall back to h5py (v7.3 / HDF5)
try:
    import scipy.io as sio
    mat = sio.loadmat(MAT_PATH)
    data = {k: v for k, v in mat.items() if not k.startswith('_')}
    print('Loaded via scipy.io.  Fields:', list(data.keys()))
    _h5 = False
except Exception as e:
    print(f'scipy.io failed ({e}), trying h5py ...')
    import h5py
    _h5_file = h5py.File(MAT_PATH, 'r')
    # h5py stores arrays transposed vs MATLAB convention — transpose back
    data = {k: np.array(_h5_file[k]).T for k in _h5_file}
    print('Loaded via h5py.  Fields:', list(data.keys()))
    _h5_file.close()
    _h5 = True

def _get_field(d, *names):
    """Case-insensitive lookup of one of several candidate field names."""
    lmap = {k.lower(): k for k in d}
    for n in names:
        if n.lower() in lmap:
            return np.array(d[lmap[n.lower()]]).squeeze().astype(np.float32)
    return None

# Extract by actual MAT field names from COBRI iSCORS output
#   BG_img   — cell morphology (mean intensity / background image)
#   Cond_map — condensation = V_DLS / D (unitless ratio)
#   D_map    — diffusion coefficient (μm²/s or px²/frame)
#   V_map    — velocity magnitude (μm/s or px/frame)
bg_ref   = _get_field(data, 'BG_img',   'bg_img',   'bg',          'background', 'Background')
cond_ref = _get_field(data, 'Cond_map', 'cond_map', 'condensation','cond')
d_ref    = _get_field(data, 'D_map',    'd_map',    'diffusion',   'Diffusion')
v_ref    = _get_field(data, 'V_map',    'v_map',    'velocity',    'Velocity')

for name, arr in [('BG_img', bg_ref), ('Cond_map', cond_ref),
                  ('D_map', d_ref),   ('V_map', v_ref)]:
    if arr is not None:
        print(f'{name:10s}: shape={arr.shape}  '
              f'range=[{arr.min():.4f}, {arr.max():.4f}]')
    else:
        print(f'{name:10s}: NOT FOUND — check fields list above and set manually')

# ── 4-panel figure of traditional iSCORS maps ────────────────────────────────
maps_to_show = [(n, a, c) for n, a, c in [
    ('BG_img  (cell morphology)',  bg_ref,   'gray'),
    ('Cond_map  (V_DLS / D)',      cond_ref, 'viridis'),
    ('D_map  (diffusion coeff)',   d_ref,    'magma'),
    ('V_map  (velocity)',          v_ref,    'plasma'),
] if a is not None]

if maps_to_show:
    fig_mat, axes_mat = plt.subplots(1, len(maps_to_show),
                                     figsize=(5 * len(maps_to_show), 5))
    if len(maps_to_show) == 1:
        axes_mat = [axes_mat]
    for ax, (title, arr, cmap) in zip(axes_mat, maps_to_show):
        fin = arr[np.isfinite(arr)]
        p1, p99 = np.percentile(fin, 1), np.percentile(fin, 99)
        im = ax.imshow(arr, cmap=cmap, vmin=p1, vmax=p99)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                     label=f'[{p1:.3f}, {p99:.3f}]')
        ax.set_title(title, fontsize=10)
        ax.axis('off')
    fig_mat.suptitle('Traditional iSCORS Output Maps  (1st–99th percentile clip)',
                     fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No maps found — check field names above.')
    fig_mat = None

# ── Save reference TIFs to Drive ─────────────────────────────────────────────
for fname, arr in [('iscors_bg_img.tif',   bg_ref),
                   ('iscors_cond_map.tif', cond_ref),
                   ('iscors_d_map.tif',    d_ref),
                   ('iscors_v_map.tif',    v_ref)]:
    if arr is not None:
        p = os.path.join(SAVE_DIR, fname)
        tifffile.imwrite(p, arr)
        print(f'Saved → {p}')

if fig_mat is not None:
    p = os.path.join(SAVE_DIR, 'iscors_traditional_maps.png')
    fig_mat.savefig(p, dpi=120, bbox_inches='tight')
    print(f'Saved → {p}')


In [ ]:
# ── Preprocess Real Video ──────────────────────────────────────────────────
# Preprocessing pipeline:
#   1. Chunked spatial binning (BIN_FACTOR × BIN_FACTOR)
#   2. Flat-field   : ÷ per-pixel temporal median
#   3. BG removal   : ÷ per-frame Gaussian-smoothed self (σ=4)
#
# Signal centering note
#   compute_g_empirical_map divides by mean_I².
#   video_proc keeps mean ≈ 1.0  → G formula is valid.
#   (video_proc − 1) is ONLY for visualisation of zero-centred fluctuations.
import numpy as np
import tifffile
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

CHUNK_SIZE = 100
N_FRAMES   = 2000
BIN_FACTOR = 2    # ← 2 for 2×2 binning, 4 for 4×4 binning

# ── Quick preview of raw frame 0 (before binning) ───────────────────────────
frame0_raw = tifffile.imread(VIDEO_PATH, key=0).astype(np.float32)
H_orig, W_orig = frame0_raw.shape
H_bin, W_bin   = H_orig // BIN_FACTOR, W_orig // BIN_FACTOR
print(f'Original frame shape : {H_orig} × {W_orig}')
print(f'After {BIN_FACTOR}×{BIN_FACTOR} binning : {H_bin} × {W_bin}')
print(f'Pixel size ratio     : ×{BIN_FACTOR} (spatial resolution reduced)')

fig, ax = plt.subplots(figsize=(6, 5))
p1, p99 = np.percentile(frame0_raw, 1), np.percentile(frame0_raw, 99)
im = ax.imshow(frame0_raw, cmap='gray', vmin=p1, vmax=p99)
plt.colorbar(im, ax=ax, label='Intensity')
ax.set_title(f'Frame 0 — raw  ({H_orig}×{W_orig})')
ax.axis('off')
plt.tight_layout()
plt.show()

# ── Chunked read + spatial binning ──────────────────────────────────────────
with tifffile.TiffFile(VIDEO_PATH) as tif:
    total_frames = len(tif.pages)

n_frames  = min(N_FRAMES, total_frames)
video_raw = np.empty((n_frames, H_bin, W_bin), dtype=np.float32)

print(f'\nReading {n_frames} frames in chunks of {CHUNK_SIZE}  '
      f'({BIN_FACTOR}×{BIN_FACTOR} binning) ...')
for start in range(0, n_frames, CHUNK_SIZE):
    end   = min(start + CHUNK_SIZE, n_frames)
    chunk = tifffile.imread(VIDEO_PATH, key=range(start, end)).astype(np.float32)
    T_c   = end - start
    video_raw[start:end] = (chunk
        .reshape(T_c, H_bin, BIN_FACTOR, W_bin, BIN_FACTOR)
        .mean(axis=(2, 4)))
    del chunk
    if start % (CHUNK_SIZE * 4) == 0:
        print(f'  frames {start:4d}–{end-1:4d}  /  {n_frames}')

T, H, W = video_raw.shape
print(f'\nLoaded (binned): T={T}  H={H}  W={W}  (float32)')
print(f'Intensity range : [{video_raw.min():.1f}, {video_raw.max():.1f}]  '
      f'mean={video_raw.mean():.1f}')

# ── Step 1: flat-field (÷ temporal median) ──────────────────────────────────
print('\nStep 1: flat-field (/ temporal median) ...')
median_xy = np.median(video_raw, axis=0)           # (H, W)
video_ff  = video_raw / (median_xy[np.newaxis] + 1e-10)
print(f'  After flat-field: mean={video_ff.mean():.4f}  std={video_ff.std():.6f}')

# ── Step 2: per-frame Gaussian background division ──────────────────────────
# sigma=4 in binned-pixel units.  Effective physical sigma = 4*BIN_FACTOR original px.
print('Step 2: Gaussian BG division (sigma=4 binned px, per frame) ...')
video_proc = np.empty_like(video_ff)
for t in range(T):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
    if t % max(1, T // 5) == 0:
        print(f'  frame {t:4d}/{T}')

print(f'\nProcessed: mean={video_proc.mean():.4f}  std={video_proc.std():.6f}  '
      f'range=[{video_proc.min():.4f}, {video_proc.max():.4f}]')
print('>>> mean ≈ 1.0  →  G(τ) denominator <I>² is valid ✓')

# ── Signal centering check ──────────────────────────────────────────────────
sig_zero = video_proc[0] - 1.0
abs_99   = np.percentile(np.abs(sig_zero), 99)
print(f'\nFluctuation frame (δI/I): mean={sig_zero.mean():.6f}  '
      f'std={sig_zero.std():.6f}  |p99|={abs_99:.6f}')

# ── 3-panel plot ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

p1r, p99r = np.percentile(video_raw[0], 1), np.percentile(video_raw[0], 99)
im0 = axes[0].imshow(video_raw[0], cmap='gray', vmin=p1r, vmax=p99r)
plt.colorbar(im0, ax=axes[0], label='Intensity')
axes[0].set_title(f'Frame 0 — raw {BIN_FACTOR}×{BIN_FACTOR} binned ({H}×{W})')
axes[0].axis('off')

p1p, p99p = np.percentile(video_proc[0], 1), np.percentile(video_proc[0], 99)
im1 = axes[1].imshow(video_proc[0], cmap='gray', vmin=p1p, vmax=p99p)
plt.colorbar(im1, ax=axes[1], label='I/<I>')
axes[1].set_title('Frame 0 — after flat-field + BG  (mean≈1, G input)')
axes[1].axis('off')

abs_99v = np.percentile(np.abs(sig_zero), 99)
im2 = axes[2].imshow(sig_zero, cmap='RdBu_r', vmin=-abs_99v, vmax=abs_99v)
plt.colorbar(im2, ax=axes[2], label='δI/I')
axes[2].set_title('Frame 0 — fluctuation δI/I  (video_proc − 1, viz only)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# ── CV distribution (cell-detection sanity check) ────────────────────────────
mean_I = video_proc.mean(axis=0)
cv_map = video_proc.std(axis=0) / (mean_I + 1e-10)
print(f'\nCV map:  mean={cv_map.mean():.4f}  '
      f'p5={np.percentile(cv_map, 5):.4f}  '
      f'p50={np.percentile(cv_map, 50):.4f}  '
      f'p95={np.percentile(cv_map, 95):.4f}')
print(f'Cell pixels (CV≥0.005): {(cv_map >= 0.005).sum()} / {cv_map.size}  '
      f'({100*(cv_map>=0.005).mean():.1f}%)')
print(f'\nPreprocessing done.  BIN_FACTOR={BIN_FACTOR}  output: {T}×{H}×{W}')


In [ ]:
# ── 10 Representative Video Frames — Compress & Download ──────────────────
# Extracts 10 evenly-spaced frames from video_proc (the preprocessed video),
# saves as a single multi-page compressed TIF, and downloads to local machine.
import numpy as np, tifffile as _tiff, os
from google.colab import files

T_total      = video_proc.shape[0]
frame_idx    = np.linspace(0, T_total - 1, 10, dtype=int)
preview      = video_proc[frame_idx].astype(np.float32)    # (10, H, W)

preview_name = f'video_preview_10frames_{VERSION}.tif'
_tiff.imwrite(preview_name, preview, compression='zlib')
kb = os.path.getsize(preview_name) / 1024
print(f'Saved {preview_name}  ({kb:.1f} KB)  frames: {frame_idx.tolist()}')
files.download(preview_name)


In [ ]:
# ── Real Video Inference ───────────────────────────────────────────────────
import sys, os, numpy as np

# Clear HuggingFace 'datasets' from cache so local package is found
for _k in list(sys.modules.keys()):
    if _k == 'datasets' or _k.startswith('datasets.'):
        del sys.modules[_k]

import torch
import matplotlib.pyplot as plt
from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
CKPT_PATH  = f'./checkpoint/pissl_phys_recon_{VERSION}.pth'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Compute G_empirical (mode=eval → full frame, no blind-spot)
# video_proc has mean≈1.0 — required for G(τ) = <δI·δI(τ)> / <I>²
print('Computing G_empirical ...')
real_ds = PhysReconDataset(
    video_tensor = video_proc,
    recon_taus   = RECON_TAUS,
    patch_size   = 64,
    mode         = 'eval',
)

# Load trained model
print(f'Loading: {CKPT_PATH}')
model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# Pad to multiple of 8 (3× MaxPool2d in encoder)
full_input = real_ds[0]                                   # (K, H, W)
K, Hv, Wv  = full_input.shape
pad_h = (8 - Hv % 8) % 8
pad_w = (8 - Wv % 8) % 8
if pad_h or pad_w:
    import torch.nn.functional as F
    full_input = F.pad(full_input, (0, pad_w, 0, pad_h))
    print(f'Padded input: {Hv}×{Wv} → {Hv+pad_h}×{Wv+pad_w}')

with torch.no_grad():
    preds_real = model(full_input.unsqueeze(0).to(device))  # (1, 2, H', W')

gamma_pred = preds_real[0, 0].cpu().numpy()[:Hv, :Wv]
alpha_pred = preds_real[0, 1].cpu().numpy()[:Hv, :Wv]
cell_mask  = real_ds.cell_mask

# ── Plot inference maps (percentile-clipped) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, data, cmap, title in [
    (axes[0], np.where(cell_mask, gamma_pred, np.nan), 'magma',  f'Model gamma [{VERSION}]'),
    (axes[1], np.where(cell_mask, alpha_pred, np.nan), 'plasma', f'Model alpha [{VERSION}]'),
]:
    p1  = np.nanpercentile(data, 1)
    p99 = np.nanpercentile(data, 99)
    im  = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, label=f'[{p1:.3f}, {p99:.3f}]')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Model Inference — Real Cell  (1st–99th percentile clip)', fontsize=13)
plt.tight_layout()
plt.show()

# ── G_norm channel plots (diagnostic: are autocorrelation channels clean?) ───
g_norm_np = full_input[:K].numpy()[:, :Hv, :Wv]    # (K, H, W) — masked G_norm
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, (ax, tau) in enumerate(zip(axes.flat, RECON_TAUS)):
    ch = np.where(cell_mask, g_norm_np[i], np.nan)
    p1, p99 = np.nanpercentile(ch, 1), np.nanpercentile(ch, 99)
    im = ax.imshow(ch, cmap='inferno', vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(f'G_norm  τ={tau}', fontsize=9)
    ax.axis('off')
plt.suptitle('G_norm channels fed to model', fontsize=12)
plt.tight_layout()
plt.show()

# ── Statistics ───────────────────────────────────────────────────────────────
g_cell = gamma_pred[cell_mask]
a_cell = alpha_pred[cell_mask]
print(f'\nCell pixels : {cell_mask.sum()}  ({100*cell_mask.mean():.1f}% of frame)')
print(f'Model gamma : mean={g_cell.mean():.3f}  std={g_cell.std():.3f}  '
      f'p1={np.percentile(g_cell,1):.3f}  p99={np.percentile(g_cell,99):.3f}')
print(f'Model alpha : mean={a_cell.mean():.3f}  std={a_cell.std():.3f}  '
      f'p1={np.percentile(a_cell,1):.3f}  p99={np.percentile(a_cell,99):.3f}')
# ── Save G_norm channel TIFs to Drive (one file per τ channel) ──────────────
import tifffile as _tiff
for _i, _tau in enumerate(RECON_TAUS):
    _ch   = g_norm_np[_i].astype(np.float32)          # (H, W) — masked G_norm
    _path = os.path.join(SAVE_DIR, f'gnorm_ch{_i:02d}_tau{_tau}.tif')
    _tiff.imwrite(_path, _ch)
print(f'Saved {len(RECON_TAUS)} G_norm channel TIFs → {SAVE_DIR}')

print('Inference done.')


In [ ]:
# ── G_norm Statistics: Real vs Synthetic vs Theory ────────────────────────
# Three questions answered:
#   1. Is G(τ=1) near zero? (normalization instability risk)
#   2. How does the empirical G_norm curve compare to the model's predicted curve?
#   3. How does real-data G_norm compare to synthetic (known γ,α) curves?
#
# G_norm theory: G_norm(τ; γ, α) = (1+γ) / (1 + γ·τ^α)
# τ=1 always gives G_norm=1 by definition (normalisation point).
# Real data: if G_empirical(τ=1) ≈ 0, G_norm becomes noisy at all channels.
import numpy as np
import matplotlib.pyplot as plt

taus      = np.array(RECON_TAUS, dtype=np.float32)
g_norm_np = real_ds.g_norm                     # (H, W, K) unnormalised G_norm
cell_mask = real_ds.cell_mask                   # (H, W) bool

# ── 1. Channel-wise mean ± std over cell pixels ───────────────────────────
g_cell  = g_norm_np[cell_mask]                  # (N_cell, K)
g_mean  = g_cell.mean(axis=0)                   # (K,)
g_std   = g_cell.std(axis=0)                    # (K,)

print('=== G_norm channel statistics (real data, cell pixels) ===')
print(f'{"τ":>6}  {"mean G_norm":>12}  {"std":>8}  {"SNR (mean/std)":>16}')
for k, (tau, gm, gs) in enumerate(zip(taus, g_mean, g_std)):
    snr = gm / (gs + 1e-10)
    flag = '  ← noisy' if snr < 2 else ''
    print(f'{int(tau):>6}  {gm:>12.4f}  {gs:>8.4f}  {snr:>16.2f}{flag}')

# ── 2. Check G_empirical(τ=1) magnitude (normalization denominator) ───────
g_empirical_tau1 = real_ds.g_norm[:, :, 0] * 1.0   # already normalised to 1 at τ=1
# Recover raw G_empirical(τ=1): g_norm = g_emp / g_emp(τ=1) → can't recover without raw
# But we can check: if real G_norm(τ=2) << 1, dynamics are very fast (G decays in τ<1)
tau2_ratio = g_norm_np[:, :, 1][cell_mask].mean()   # G_norm(τ=2) / G_norm(τ=1)
print(f'\nG_norm(τ=2) / G_norm(τ=1) mean over cell: {tau2_ratio:.4f}')
if tau2_ratio < 0.5:
    print('WARNING: G_norm(τ=2) < 0.5 → dynamics faster than τ=1 lag;'
          ' G(τ=1) near zero → small-τ channels are noise-dominated.')
elif tau2_ratio < 0.8:
    print('CAUTION: Fast dynamics. Some small-τ channel noise expected.')
else:
    print('OK: G_norm decays slowly — small-τ channels should be stable.')

# ── 3. Compare to theory at mean (γ, α) and synthetic region curves ───────
def g_norm_theory(taus, gamma, alpha):
    return (1.0 + gamma) / (1.0 + gamma * taus ** alpha)

tau_plot = np.geomspace(1, taus[-1], 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: real data mean ± std with theoretical overlay from model predictions
ax = axes[0]
ax.fill_between(taus, g_mean - g_std, g_mean + g_std,
                alpha=0.25, color='steelblue', label='real ±1σ')
ax.plot(taus, g_mean, 'o-', color='steelblue', lw=2, label='real mean')

# Theory from mean model prediction
g_m = gamma_pred[cell_mask].mean()
a_m = alpha_pred[cell_mask].mean()
ax.plot(tau_plot, g_norm_theory(tau_plot, g_m, a_m), 'r--', lw=1.5,
        label=f'theory γ={g_m:.3f} α={a_m:.3f}  (model mean)')
ax.set_xscale('log')
ax.set_xlabel('τ (lag)')
ax.set_ylabel('G_norm(τ)')
ax.set_title('Real data: empirical G_norm vs model-predicted theory')
ax.legend(fontsize=8)
ax.set_ylim(-0.1, 1.4)
ax.axhline(1.0, color='k', lw=0.5, ls=':')
ax.axhline(0.0, color='k', lw=0.5, ls=':')

# Right: synthetic region reference curves (GT known)
ax2 = axes[1]
ax2.plot(taus, g_mean, 'o-', color='steelblue', lw=2, label='real mean')
synth_regions = [
    (0.10, 1.0,  'green',  'synth cell body  γ=0.10 α=1.0'),
    (0.50, 1.5,  'red',    'synth fast spot  γ=0.50 α=1.5'),
    (0.05, 0.5,  'purple', 'synth slow spot  γ=0.05 α=0.5'),
]
for gamma, alpha, color, label in synth_regions:
    ax2.plot(tau_plot, g_norm_theory(tau_plot, gamma, alpha),
             '--', color=color, lw=1.5, label=label)
ax2.set_xscale('log')
ax2.set_xlabel('τ (lag)')
ax2.set_ylabel('G_norm(τ)')
ax2.set_title('Real data G_norm vs synthetic region curves')
ax2.legend(fontsize=8)
ax2.set_ylim(-0.1, 1.4)
ax2.axhline(1.0, color='k', lw=0.5, ls=':')
ax2.axhline(0.0, color='k', lw=0.5, ls=':')

plt.tight_layout()
plt.suptitle('G_norm Curve Analysis — Real vs Theory vs Synthetic', fontsize=13, y=1.02)
plt.show()

p = os.path.join(SAVE_DIR, f'gnorm_stats_{VERSION}.png')
fig.savefig(p, dpi=120, bbox_inches='tight')
print(f'\nSaved → {p}')


In [ ]:
# ── Compare Model vs Traditional iSCORS + Save Results ─────────────────────
# Variables expected from previous cells:
#   gamma_pred, alpha_pred, cell_mask   (from real-inference)
#   bg_ref, cond_ref, d_ref, v_ref      (from real-mat; any may be None)
#   g_cell, a_cell, T, VIDEO_PATH       (from real-inference)
import numpy as np
import matplotlib.pyplot as plt
import tifffile, os
from skimage.transform import resize as sk_resize

cell_mask = real_ds.cell_mask

def _match(arr, shape):
    if arr.shape == shape:
        return arr
    return sk_resize(arr, shape, preserve_range=True,
                     anti_aliasing=True).astype(np.float32)

def _pclip(data, lo=1, hi=99):
    v = data[np.isfinite(data)]
    return (np.percentile(v, lo), np.percentile(v, hi))

gm_data = np.where(cell_mask, gamma_pred, np.nan)
am_data = np.where(cell_mask, alpha_pred, np.nan)

# ── Section 1: Model output maps ─────────────────────────────────────────────
fig1, axes1 = plt.subplots(1, 2, figsize=(10, 5))
for ax, data, cmap, title in [
    (axes1[0], gm_data, 'magma',  f'Model gamma [{VERSION}]'),
    (axes1[1], am_data, 'plasma', f'Model alpha [{VERSION}]'),
]:
    p1, p99 = _pclip(data)
    im = ax.imshow(data, cmap=cmap, vmin=p1, vmax=p99)
    plt.colorbar(im, ax=ax, label=f'[{p1:.3f}, {p99:.3f}]')
    ax.set_title(title, fontsize=11)
    ax.axis('off')
fig1.suptitle('Model Predictions  (1st–99th percentile clip)', fontsize=13)
plt.tight_layout()
plt.show()

# ── Section 2: Model gamma vs 1/D_map ────────────────────────────────────────
# Traditional iSCORS D_map measures diffusion coefficient.
# Our gamma encodes decay speed: high gamma → fast decay → small D.
# Expected relationship: gamma ∝ 1/D_map  (inverse, not identity).
fig2 = fig3 = None
d_matched = inv_d = None

if d_ref is not None:
    d_matched = _match(d_ref, gamma_pred.shape)
    inv_d     = 1.0 / (d_matched + 1e-10)
    inv_d_masked = np.where(cell_mask, inv_d, np.nan)

    fig2, axes2 = plt.subplots(1, 3, figsize=(15, 5))

    # Panel 1 — model gamma
    p1g, p99g = _pclip(gm_data)
    im = axes2[0].imshow(gm_data, cmap='magma', vmin=p1g, vmax=p99g)
    plt.colorbar(im, ax=axes2[0], label=f'[{p1g:.3f}, {p99g:.3f}]')
    axes2[0].set_title(f'Model gamma [{VERSION}]', fontsize=10)
    axes2[0].axis('off')

    # Panel 2 — D_map (reversed colormap: high D → fast → same direction as high gamma)
    p1d, p99d = _pclip(d_matched[np.isfinite(d_matched)])
    im = axes2[1].imshow(d_matched, cmap='magma_r', vmin=p1d, vmax=p99d)
    plt.colorbar(im, ax=axes2[1], label=f'[{p1d:.3f}, {p99d:.3f}]')
    axes2[1].set_title('trad. D_map  (reversed: high=dark → slow)', fontsize=10)
    axes2[1].axis('off')

    # Panel 3 — 1/D_map (direct comparison with gamma, same colormap direction)
    p1i, p99i = _pclip(inv_d_masked)
    im = axes2[2].imshow(inv_d_masked, cmap='magma', vmin=p1i, vmax=p99i)
    plt.colorbar(im, ax=axes2[2], label=f'[{p1i:.4f}, {p99i:.4f}]')
    axes2[2].set_title('1/D_map  (∝ gamma, same scale direction)', fontsize=10)
    axes2[2].axis('off')

    fig2.suptitle('Model gamma vs Traditional D_map  (both: bright = fast diffusion)',
                  fontsize=12)
    plt.tight_layout()
    plt.show()

# ── Section 3: Supplementary traditional maps ─────────────────────────────────
supp = [(n, a, c) for n, a, c in [
    ('BG_img  (morphology)', bg_ref,   'gray'),
    ('Cond_map  (V/D)',       cond_ref, 'viridis'),
    ('V_map  (velocity)',     v_ref,    'plasma'),
] if a is not None]

if supp:
    fig3, axes3 = plt.subplots(1, len(supp), figsize=(5 * len(supp), 5))
    if len(supp) == 1:
        axes3 = [axes3]
    for ax, (title, arr, cmap) in zip(axes3, supp):
        matched = _match(arr, gamma_pred.shape)
        p1, p99 = _pclip(matched[np.isfinite(matched)])
        im = ax.imshow(matched, cmap=cmap, vmin=p1, vmax=p99)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                     label=f'[{p1:.3f}, {p99:.3f}]')
        ax.set_title(f'iSCORS {title}', fontsize=10)
        ax.axis('off')
    fig3.suptitle('Traditional iSCORS Supplementary Maps', fontsize=12)
    plt.tight_layout()
    plt.show()

# ── Section 4: Scatter — model gamma vs 1/D_map ──────────────────────────────
if d_matched is not None:
    from scipy.stats import pearsonr, spearmanr
    g_c     = gamma_pred[cell_mask]
    inv_d_c = inv_d[cell_mask]
    r_p, _  = pearsonr(g_c,  inv_d_c)
    r_s, _  = spearmanr(g_c, inv_d_c)

    fig4, ax4 = plt.subplots(figsize=(5, 5))
    sc = ax4.scatter(inv_d_c, g_c, s=0.5, alpha=0.2, c=g_c, cmap='magma')
    plt.colorbar(sc, ax=ax4, label='model gamma')
    ax4.set_xlabel('1/D_map  (traditional iSCORS)')
    ax4.set_ylabel('Model gamma')
    ax4.set_title(f'Pearson r={r_p:.3f}   Spearman r={r_s:.3f}')
    plt.tight_layout()
    plt.show()
    print(f'\nCorrelation gamma ~ 1/D:  Pearson={r_p:.4f}  Spearman={r_s:.4f}')
else:
    print('D_map not found — scatter plot skipped.')

# ── Section 5: Numerical stats ───────────────────────────────────────────────
g_cell = gamma_pred[cell_mask]
a_cell = alpha_pred[cell_mask]
stats_lines = [
    f'=== Real Cell Analysis [{VERSION}] ===',
    f'Video      : {VIDEO_PATH}',
    f'Frames used: {T}   Cell pixels: {cell_mask.sum()}  ({100*cell_mask.mean():.1f}%)',
    '',
    '--- Model predictions ---',
    f'  gamma : mean={g_cell.mean():.4f}  std={g_cell.std():.4f}  '
    f'p1={np.percentile(g_cell,1):.4f}  p99={np.percentile(g_cell,99):.4f}',
    f'  alpha : mean={a_cell.mean():.4f}  std={a_cell.std():.4f}  '
    f'p1={np.percentile(a_cell,1):.4f}  p99={np.percentile(a_cell,99):.4f}',
]
if d_matched is not None:
    d_cell = d_matched[cell_mask]
    stats_lines += [
        '',
        '--- vs Traditional iSCORS D_map ---',
        f'  D_map     : mean={d_cell.mean():.4f}  '
        f'p1={np.percentile(d_cell,1):.4f}  p99={np.percentile(d_cell,99):.4f}',
        f'  Pearson r(gamma, 1/D): {r_p:.4f}   Spearman: {r_s:.4f}',
        '  (Note: alpha has no equivalent in traditional iSCORS output)',
    ]
print('\n'.join(stats_lines))

# ── Section 6: Save results to Drive ─────────────────────────────────────────
os.makedirs(SAVE_DIR, exist_ok=True)

tifffile.imwrite(os.path.join(SAVE_DIR, f'model_gamma_{VERSION}.tif'),
                 np.where(cell_mask, gamma_pred, 0).astype(np.float32))
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_alpha_{VERSION}.tif'),
                 np.where(cell_mask, alpha_pred, 0).astype(np.float32))

_stats_path = os.path.join(SAVE_DIR, f'real_stats_{VERSION}.txt')
with open(_stats_path, 'w') as f:
    f.write('\n'.join(stats_lines) + '\n')

fig1.savefig(os.path.join(SAVE_DIR, f'model_maps_{VERSION}.png'),
             dpi=120, bbox_inches='tight')
if fig2 is not None:
    fig2.savefig(os.path.join(SAVE_DIR, f'gamma_vs_D_{VERSION}.png'),
                 dpi=120, bbox_inches='tight')
if fig3 is not None:
    fig3.savefig(os.path.join(SAVE_DIR, f'iscors_supp_maps_{VERSION}.png'),
                 dpi=120, bbox_inches='tight')

print(f'\nSaved → {SAVE_DIR}/')
for fname in [f'model_gamma_{VERSION}.tif', f'model_alpha_{VERSION}.tif',
              f'real_stats_{VERSION}.txt',  f'model_maps_{VERSION}.png',
              f'gamma_vs_D_{VERSION}.png',  f'iscors_supp_maps_{VERSION}.png']:
    p = os.path.join(SAVE_DIR, fname)
    if os.path.exists(p):
        print(f'  {fname}')


In [ ]:
# ── Download Real Inference Results as ZIP ──────────────────────────────────
import zipfile, os, glob
from google.colab import files

zip_name = f'inference_real_{VERSION}.zip'

collected = []
for pattern in [
    os.path.join(SAVE_DIR, f'*{VERSION}*'),
    os.path.join(SAVE_DIR, 'iscors_*.tif'),
    os.path.join(SAVE_DIR, 'iscors_traditional_maps.png'),
    os.path.join(SAVE_DIR, f'gnorm_stats_{VERSION}.png'),
]:
    collected.extend(glob.glob(pattern))

collected = sorted(set(collected))
print(f'Files to zip ({len(collected)}):')
for p in collected:
    print(f'  {os.path.basename(p)}  ({os.path.getsize(p)/1024:.1f} KB)')

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in collected:
        zf.write(p, os.path.basename(p))

print(f'\nCreated {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)
